# Final Course Scoring Model

Each course receives a recommendation score based on multiple factors.

$$
CourseUtility =
w_1 \cdot SkillCoverageScore
+ w_2 \cdot DifficultyScore
+ w_3 \cdot QualityScore
- w_4 \cdot TimePenalty
$$

Recommended weights:

- SkillCoverageScore: **0.55**
- DifficultyScore: **0.10**
- QualityScore: **0.15**
- TimePenalty: **0.20**


# Skill Coverage Score

The skill coverage score measures how well a course addresses the user's **missing skills**.

For each missing skill:

1. Compute semantic similarity between the missing skill and course skills  
2. Select the **best-matching course skill**  
3. Weight the similarity by the skill's importance  

The score is calculated as:

$$
SkillCoverageScore =
\frac{
\sum_{i=1}^{n} importance_i \cdot similarity_i
}{
\sum_{i=1}^{n} importance_i
}
$$

Courses covering **multiple high-priority missing skills** will receive higher scores.

# Difficulty Matching Score

Course difficulty should align with the user's current skill level.

Difficulty levels are mapped to numeric values:

- Beginner = 1  
- Intermediate = 2  
- Advanced = 3  

The score is defined as:

$$
DifficultyScore =
\max\left(
0,\,
1 - 0.4 \cdot |course\_level - user\_level|
\right)
$$

This allows small differences while penalizing large difficulty gaps.

# Course Quality Score

Course quality is evaluated using ratings and review counts.

Both variables are normalized and combined:

$$
QualityScore =
0.7 \cdot RatingNorm
+
0.3 \cdot ReviewNorm
$$

- Ratings reflect course quality  
- Review counts represent credibility and popularity

# Time Penalty

Instead of rewarding courses close to a target duration, time is treated as a constraint.

The time penalty is defined as:

$$
TimePenalty =
\frac{course\_hours}{time\_budget}
$$

If a course exceeds the total time budget:

$$
TimePenalty = 1.5
$$

This strongly penalizes courses that consume too much of the user's available learning time.

# Efficiency Score

To prioritize efficient learning, each course is evaluated by its **utility per hour**.

$$
ValuePerHour =
\frac{CourseUtility}{course\_hours}
$$

Courses that provide **greater skill coverage within less time** will receive higher efficiency scores.

In [ ]:
import pulp

# --- 1. Utility Scoring Logic
def calculate_utility(course, user_level, time_budget):
    # w1*SkillCoverage + w2*Difficulty + w3*Quality - w4*TimePenalty
    w1, w2, w3, w4 = 0.55, 0.10, 0.15, 0.20
    
    # Difficulty Score logic
    diff_score = max(0, 1 - 0.4 * abs(course['level'] - user_level))
    
    # Quality Score logic
    quality_score = (0.7 * course['rating_norm']) + (0.3 * course['review_norm'])
    
    # Time Penalty
    time_penalty = course['hours'] / time_budget if course['hours'] <= time_budget else 1.5
    
    # For this example, we assume SkillCoverageScore is pre-calculated
    utility = (w1 * course['skill_score']) + (w2 * diff_score) + (w3 * quality_score) - (w4 * time_penalty)
    return utility

# --- 2. Data Setup ---
time_limit = 10
user_lvl = 1 # Beginner
course_data = {
    "Course_A": {"hours": 6, "skill_score": 0.9, "level": 1, "rating_norm": 0.8, "review_norm": 0.9},
    "Course_B": {"hours": 4, "skill_score": 0.7, "level": 2, "rating_norm": 0.9, "review_norm": 0.7},
    "Course_C": {"hours": 10, "skill_score": 0.95, "level": 1, "rating_norm": 0.9, "review_norm": 0.8}
}

# Pre-calculate utility for each
for name, data in course_data.items():
    data['final_utility'] = calculate_utility(data, user_lvl, time_limit)

# --- 3. The Optimization Model ---
prob = pulp.LpProblem("Maximize_Learning_Efficiency", pulp.LpMaximize)

# Decision Variables: 1 if we take the course, 0 otherwise
select_vars = pulp.LpVariable.dicts("Select", course_data.keys(), cat='Binary')

# Objective: Maximize Total Utility
prob += pulp.lpSum([course_data[i]['final_utility'] * select_vars[i] for i in course_data.keys()])

# Constraint: Sum of hours <= time_budget
prob += pulp.lpSum([course_data[i]['hours'] * select_vars[i] for i in course_data.keys()]) <= time_limit

# Solve
prob.solve()

# --- 4. Output Results ---
print(f"Optimization Status: {pulp.LpStatus[prob.status]}")
for i in course_data.keys():
    if pulp.value(select_vars[i]) == 1:
        print(f"Selected: {i} | Utility: {course_data[i]['final_utility']:.2f} | Time: {course_data[i]['hours']}h")

ModuleNotFoundError: No module named 'pulp'

In [4]:
import pandas as pd

# Load the JSONL file
df = pd.read_json(r"C:\Users\oyun_\Downloads\final_gap_analysis (2).jsonl", lines=True)

In [15]:
df["skill_extracted"][10]

{'Focused': {'priority_score': 3.0, 'difficulty_level': 'Intermediate'},
 'Cpa': {'priority_score': 7.2, 'difficulty_level': 'Intermediate'},
 'Microsoft': {'priority_score': 7.2, 'difficulty_level': 'Intermediate'},
 'Filing': {'priority_score': 7.2, 'difficulty_level': 'Intermediate'},
 'Tax Management': {'priority_score': 3.0, 'difficulty_level': 'Intermediate'},
 'Organization': {'priority_score': 7.2, 'difficulty_level': 'Intermediate'},
 'Communication Skills': {'priority_score': 7.2,
  'difficulty_level': 'Intermediate'},
 'Software Applications': {'priority_score': 7.2,
  'difficulty_level': 'Intermediate'},
 'Sales': {'priority_score': 3.0, 'difficulty_level': 'Intermediate'},
 'Compliance': {'priority_score': 3.0, 'difficulty_level': 'Intermediate'},
 'Strong Work Ethic': {'priority_score': 7.2,
  'difficulty_level': 'Intermediate'},
 'Finance': {'priority_score': 7.2, 'difficulty_level': 'Intermediate'},
 'Enterprise Systems': {'priority_score': 7.2,
  'difficulty_level': 'I

In [8]:
df

,job_description_text,resume_text,skill_extracted
0,Net2Source Inc. is an award-winning total work...,SummaryHighly motivated Sales Associate with e...,"{'Sql': {'priority_score': 4.8, 'difficulty_le..."
1,At Salas OBrien we tell our clients that were ...,Professional SummaryCurrently working with Cat...,"{'Excel': {'priority_score': 7.2, 'difficulty_..."
2,Schweitzer Engineering Laboratories (SEL) Infr...,SummaryI started my construction career in Jun...,"{'Python': {'priority_score': 3.0, 'difficulty..."
3,"Mizick Miller & Company, Inc. is looking for a...",SummaryCertified Electrical Foremanwith thirte...,"{'Microsoft': {'priority_score': 5.4, 'difficu..."
4,Life at Capgemini\nCapgemini supports all aspe...,SummaryWith extensive experience in business/r...,"{'R': {'priority_score': 9.0, 'difficulty_leve..."
...,...,...,...
495,Software Engineer Contract-to-hire\nBrooksourc...,Summary*11 years analog/digital circuit design...,"{'Java': {'priority_score': 5.2, 'difficulty_l..."
496,Job Description\nEssential Duties and Responsi...,SummaryAn electrical engineer with experience ...,"{'Tax': {'priority_score': 3.0, 'difficulty_le..."
497,If you can handle the accounting responsibilit...,Professional ProfileHighly motivated Sales Ass...,"{'Excel': {'priority_score': 6.0, 'difficulty_..."
498,Data EngineerLength of Assignment: 6+ monthsWo...,SummaryHardworking Senior Accountant proficien...,"{'Spark': {'priority_score': 6.0, 'difficulty_..."


In [21]:
coursera = pd.read_csv(r"C:\Users\oyun_\Downloads\coursera_full_data_Mar10.csv")


In [26]:
coursera

,Course title,Course_Type,Organization,Skills,Workload,Difficulty_level,Language,Course_link,Ratings,Review count,Payment_Model
0,SPSS: Apply & Interpret Logistic Regression Mo...,Individual Course,EDUCBA,"Logistic Regression, Model Evaluation",NaN,NaN,en,https://www.coursera.org/learn/spss-apply-inte...,5.0,13,Free Audit / Pay for Cert
1,Securing Compute Engine Applications and Resou...,Individual Course,Google Cloud,"Cloud Computing, Computer Security And Networks",1 hour 30 minutes,Intermediate,en,https://www.coursera.org/learn/googlecloud-sec...,NaN,0,Free Audit / Pay for Cert
2,Getting started with the Vertex AI Gemini 1.5 ...,Individual Course,Google Cloud,"Cloud Computing, Machine Learning",1 hour 30 minutes,Beginner,en,https://www.coursera.org/learn/googlecloud-get...,NaN,0,Free Audit / Pay for Cert
3,Overcoming Challenges in Self and Society,Individual Course,University of Colorado System,"Adaptability, Community Development, Compassio...","4 weeks of study, 2-4 hours a week",Beginner,en,https://www.coursera.org/learn/overcoming-chal...,NaN,1,Free Audit / Pay for Cert
4,Intégrer des applications dans votre Dashboard...,Individual Course,Coursera,"Marketing, Design And Product",2 heures,Beginner,fr,https://www.coursera.org/learn/integrer-applic...,NaN,0,Free Audit / Pay for Cert
...,...,...,...,...,...,...,...,...,...,...,...
19453,The Korean Alphabet: An Introduction to Hangeul,Individual Course,Sungkyunkwan University,"Ancient History, Language Competency, Language...",4 hours per week,Beginner,en,https://www.coursera.org/learn/the-korean-alph...,4.7,506,Free Audit / Pay for Cert
19454,Shape and Property Control of Metals I & II,Individual Course,Arizona State University,"Engineering Calculations, Failure Analysis, Ma...",NaN,Beginner,en,https://www.coursera.org/learn/shape-and-prope...,4.9,22,Free Audit / Pay for Cert
19455,Looker Functions and Operators,Individual Course,Google Cloud,"Cloud Computing, Software Development",1 hour,Beginner,en,https://www.coursera.org/learn/googlecloud-loo...,NaN,0,Free Audit / Pay for Cert
19456,Causal Inference,Individual Course,Columbia University,"Data Analysis, Graph Theory, Quantitative Rese...","6 weeks of study, 3-5 hours per week",Advanced,en,https://www.coursera.org/learn/causal-inference,NaN,570,Free Audit / Pay for Cert
